# Lab | Data Cleaning and Formatting

This notebook cleans and formats the insurance customer dataset, removes duplicates, saves the cleaned CSV, and performs the bonus analysis.

## Load the data

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
df = pd.read_csv(url)
print(df.head())
print("Original dimensions:", df.shape)

## Exercise 1: Cleaning column names

Column names are converted to lowercase, spaces become underscores, and ST becomes state.

In [ ]:
df.columns = (
    df.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
)
df = df.rename(columns={"st": "state"})
print(df.columns.tolist())

## Exercise 2: Cleaning invalid values

In [ ]:
df["gender"] = df["gender"].replace(
    {"Femal": "F", "female": "F", "Male": "M"}
)
df["state"] = df["state"].replace(
    {"AZ": "Arizona", "Cali": "California", "WA": "Washington"}
)
df["education"] = df["education"].replace({"Bachelors": "Bachelor"})
df["customer_lifetime_value"] = (
    df["customer_lifetime_value"].astype("string")
    .str.replace("%", "", regex=False)
)
df["vehicle_class"] = df["vehicle_class"].replace(
    {
        "Sports Car": "Luxury",
        "Luxury SUV": "Luxury",
        "Luxury Car": "Luxury",
    }
)
print("Gender values:", df["gender"].dropna().unique())
print("State values:", df["state"].dropna().unique())
print("Vehicle classes:", df["vehicle_class"].dropna().unique())

## Exercise 3: Formatting data types

Customer Lifetime Value becomes numeric. Complaint values such as 1/5/00 are converted to 5 by taking the middle value.

In [ ]:
df["customer_lifetime_value"] = pd.to_numeric(
    df["customer_lifetime_value"], errors="coerce"
)

complaints = df["number_of_open_complaints"].astype("string")
df["number_of_open_complaints"] = pd.to_numeric(
    complaints.str.split("/").str[1], errors="coerce"
)
print(df.dtypes)
print("Complaint values:", df["number_of_open_complaints"].unique())

## Exercise 4: Dealing with null values

First, completely blank rows are removed. For remaining missing values, numeric columns use their median and categorical columns use their mode. This keeps the dataset size while using robust, explainable replacement values.

In [ ]:
df = df.dropna(how="all").copy()
print("Null values before filling:\n", df.isna().sum())

numeric_columns = df.select_dtypes(include="number").columns
categorical_columns = df.select_dtypes(exclude="number").columns

for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

for column in categorical_columns:
    if df[column].isna().any():
        df[column] = df[column].fillna(df[column].mode()[0])

# The exercise asks for all numeric variables to be integers at the end.
df[numeric_columns] = df[numeric_columns].round().astype(int)

print("\nNull values after filling:\n", df.isna().sum())
print("\nNumeric data types after formatting:\n", df[numeric_columns].dtypes)

## Exercise 5: Dealing with duplicates

Duplicate rows are removed and the index is reset. The cleaned dataset is saved as a new CSV file.

In [ ]:
duplicate_count = df.duplicated().sum()
print("Duplicate rows before removal:", duplicate_count)

df = df.drop_duplicates().reset_index(drop=True)
print("Duplicate rows after removal:", df.duplicated().sum())
print("Final dimensions:", df.shape)

cleaned_csv_path = "insurance_customers_clean.csv"
df.to_csv(cleaned_csv_path, index=False)
print(f"Saved cleaned dataset to {cleaned_csv_path}")

# Bonus: Challenge 2 — reusable cleaning functions

The functions are stored in data_cleaning_functions.py. The complete pipeline is imported and run below.

In [ ]:
from data_cleaning_functions import clean_from_source

modular_df = clean_from_source(url)
print(modular_df.head())
print("Modular pipeline dimensions:", modular_df.shape)
print("Remaining null values:", modular_df.isna().sum().sum())

# Bonus: Challenge 3 — analyzing the cleaned data

High claims are defined as values above the 75th percentile. Low customer lifetime value is defined as values below the 25th percentile.

In [ ]:
claim_75th_percentile = df["total_claim_amount"].quantile(0.75)
clv_25th_percentile = df["customer_lifetime_value"].quantile(0.25)

high_claim_low_clv = df.loc[
    (df["total_claim_amount"] > claim_75th_percentile)
    & (df["customer_lifetime_value"] < clv_25th_percentile)
].copy()

print("75th-percentile claim threshold:", claim_75th_percentile)
print("25th-percentile CLV threshold:", clv_25th_percentile)
print("Number of high-claim, low-CLV customers:", len(high_claim_low_clv))
print(high_claim_low_clv[["customer", "customer_lifetime_value", "total_claim_amount"]].head())
print("\nSummary statistics:")
print(high_claim_low_clv[["customer_lifetime_value", "total_claim_amount"]].describe())